# 🩺 Multiple Disease Prediction System
### Diabetes • Heart Disease • Parkinson's Disease — Machine Learning Pipeline

This notebook builds, trains, evaluates, and saves ML models for **three disease detection tasks**:

1. **Diabetes Prediction** — PIMA Indians Diabetes Dataset
2. **Heart Disease Prediction** — Kaggle Heart Failure Prediction Dataset (Cleveland/UCI-derived)
3. **Parkinson's Disease Prediction** — UCI Oxford Parkinson's Disease Detection Dataset

All datasets are loaded **directly from public URLs** — no manual download needed. Just run the cells top to bottom (works in Jupyter or Google Colab).

**Pipeline for each disease:**
- Load data from URL
- Exploratory Data Analysis (EDA)
- Preprocessing (cleaning, encoding, scaling)
- Train/test split
- Train multiple ML models & compare
- Evaluate (accuracy, confusion matrix, classification report)
- Save the best model as a `.pkl` file (ready for a Flask/Streamlit app later)



In [ ]:
# ============================================================
# 0. INSTALL & IMPORT LIBRARIES
# ============================================================
# Uncomment the line below if running in a fresh environment (e.g. Colab)
# !pip install -q pandas numpy matplotlib seaborn scikit-learn joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                              ConfusionMatrixDisplay, roc_curve, roc_auc_score)

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_STATE = 42
print("Libraries loaded successfully.")

---
## 1️⃣ Diabetes Prediction

**Dataset:** PIMA Indians Diabetes Dataset (768 female patients, 8 clinical features)
**Source URL:** `https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv`
**Target:** `class` → 0 = No Diabetes, 1 = Diabetes


In [ ]:
# ----- 1.1 Load Data -----
diabetes_url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
diabetes_cols = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
                  'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']

diabetes_df = pd.read_csv(diabetes_url, names=diabetes_cols)
print("Shape:", diabetes_df.shape)
diabetes_df.head()

In [ ]:
# ----- 1.2 Exploratory Data Analysis -----
diabetes_df.info()
print("\nMissing values:\n", diabetes_df.isnull().sum())
print("\nClass balance:\n", diabetes_df['Outcome'].value_counts())

diabetes_df.describe()

In [ ]:
# ----- 1.3 Data Cleaning -----
# Columns where 0 is medically impossible => treat as missing, then impute with median
zero_not_allowed = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
diabetes_clean = diabetes_df.copy()
diabetes_clean[zero_not_allowed] = diabetes_clean[zero_not_allowed].replace(0, np.nan)
for col in zero_not_allowed:
    diabetes_clean[col] = diabetes_clean[col].fillna(diabetes_clean[col].median())

diabetes_clean.describe()

In [ ]:
# ----- 1.4 Visualizations -----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(x='Outcome', data=diabetes_clean, ax=axes[0])
axes[0].set_title("Diabetes Outcome Distribution")

sns.heatmap(diabetes_clean.corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=axes[1])
axes[1].set_title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
# ----- 1.5 Train/Test Split & Scaling -----
X_dia = diabetes_clean.drop('Outcome', axis=1)
y_dia = diabetes_clean['Outcome']

X_dia_train, X_dia_test, y_dia_train, y_dia_test = train_test_split(
    X_dia, y_dia, test_size=0.2, random_state=RANDOM_STATE, stratify=y_dia)

scaler_dia = StandardScaler()
X_dia_train_s = scaler_dia.fit_transform(X_dia_train)
X_dia_test_s = scaler_dia.transform(X_dia_test)

print("Train size:", X_dia_train.shape, "| Test size:", X_dia_test.shape)

In [ ]:
# ----- 1.6 Train & Compare Multiple Models -----
dia_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
    "SVM": SVC(probability=True, random_state=RANDOM_STATE),
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB()
}

dia_results = {}
for name, model in dia_models.items():
    model.fit(X_dia_train_s, y_dia_train)
    preds = model.predict(X_dia_test_s)
    acc = accuracy_score(y_dia_test, preds)
    dia_results[name] = acc
    print(f"{name:20s} -> Accuracy: {acc:.4f}")

best_dia_name = max(dia_results, key=dia_results.get)
best_dia_model = dia_models[best_dia_name]
print(f"\n🏆 Best Model: {best_dia_name} ({dia_results[best_dia_name]:.4f})")

In [ ]:
# ----- 1.7 Evaluate Best Model -----
dia_preds = best_dia_model.predict(X_dia_test_s)
print(classification_report(y_dia_test, dia_preds, target_names=["No Diabetes", "Diabetes"]))

cm = confusion_matrix(y_dia_test, dia_preds)
ConfusionMatrixDisplay(cm, display_labels=["No Diabetes", "Diabetes"]).plot(cmap="Blues")
plt.title(f"Confusion Matrix - {best_dia_name} (Diabetes)")
plt.show()

In [ ]:
# ----- 1.8 Save Model -----
with open("diabetes_model.pkl", "wb") as f:
    pickle.dump(best_dia_model, f)
with open("diabetes_scaler.pkl", "wb") as f:
    pickle.dump(scaler_dia, f)

print("Saved: diabetes_model.pkl, diabetes_scaler.pkl")

---
## 2️⃣ Heart Disease Prediction

**Dataset:** Heart Failure Prediction Dataset (918 patients, 11 clinical features)
**Source URL:** `https://raw.githubusercontent.com/akarshsnair/Dataset-cart/main/heart.csv`
**Target:** `HeartDisease` → 0 = No Disease, 1 = Disease


In [ ]:
# ----- 2.1 Load Data -----
heart_url = "https://raw.githubusercontent.com/akarshsnair/Dataset-cart/main/heart.csv"
heart_df = pd.read_csv(heart_url)
print("Shape:", heart_df.shape)
heart_df.head()

In [ ]:
# ----- 2.2 Exploratory Data Analysis -----
heart_df.info()
print("\nMissing values:\n", heart_df.isnull().sum())
print("\nClass balance:\n", heart_df['HeartDisease'].value_counts())

In [ ]:
# ----- 2.3 Encode Categorical Columns -----
heart_clean = heart_df.copy()
categorical_cols = heart_clean.select_dtypes(include='object').columns.tolist()
print("Categorical columns:", categorical_cols)

# A few rows have blank/missing cells in this dataset; impute before encoding/scaling
numeric_cols = [c for c in heart_clean.columns if c not in categorical_cols + ['HeartDisease']]
for col in numeric_cols:
    heart_clean[col] = heart_clean[col].fillna(heart_clean[col].median())
for col in categorical_cols:
    heart_clean[col] = heart_clean[col].fillna(heart_clean[col].mode().iloc[0])

encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    heart_clean[col] = le.fit_transform(heart_clean[col])
    encoders[col] = le

heart_clean.head()

In [ ]:
# ----- 2.4 Visualizations -----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(x='HeartDisease', data=heart_clean, ax=axes[0])
axes[0].set_title("Heart Disease Outcome Distribution")

sns.heatmap(heart_clean.corr(), cmap="coolwarm", ax=axes[1])
axes[1].set_title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
# ----- 2.5 Train/Test Split & Scaling -----
X_heart = heart_clean.drop('HeartDisease', axis=1)
y_heart = heart_clean['HeartDisease']

X_heart_train, X_heart_test, y_heart_train, y_heart_test = train_test_split(
    X_heart, y_heart, test_size=0.2, random_state=RANDOM_STATE, stratify=y_heart)

scaler_heart = StandardScaler()
X_heart_train_s = scaler_heart.fit_transform(X_heart_train)
X_heart_test_s = scaler_heart.transform(X_heart_test)

print("Train size:", X_heart_train.shape, "| Test size:", X_heart_test.shape)

In [ ]:
# ----- 2.6 Train & Compare Multiple Models -----
heart_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
    "SVM": SVC(probability=True, random_state=RANDOM_STATE),
    "KNN": KNeighborsClassifier(),
}

heart_results = {}
for name, model in heart_models.items():
    model.fit(X_heart_train_s, y_heart_train)
    preds = model.predict(X_heart_test_s)
    acc = accuracy_score(y_heart_test, preds)
    heart_results[name] = acc
    print(f"{name:20s} -> Accuracy: {acc:.4f}")

best_heart_name = max(heart_results, key=heart_results.get)
best_heart_model = heart_models[best_heart_name]
print(f"\n🏆 Best Model: {best_heart_name} ({heart_results[best_heart_name]:.4f})")

In [ ]:
# ----- 2.7 Evaluate Best Model -----
heart_preds = best_heart_model.predict(X_heart_test_s)
print(classification_report(y_heart_test, heart_preds, target_names=["No Disease", "Disease"]))

cm = confusion_matrix(y_heart_test, heart_preds)
ConfusionMatrixDisplay(cm, display_labels=["No Disease", "Disease"]).plot(cmap="Greens")
plt.title(f"Confusion Matrix - {best_heart_name} (Heart Disease)")
plt.show()

In [ ]:
# ----- 2.8 Save Model -----
with open("heart_model.pkl", "wb") as f:
    pickle.dump(best_heart_model, f)
with open("heart_scaler.pkl", "wb") as f:
    pickle.dump(scaler_heart, f)
with open("heart_encoders.pkl", "wb") as f:
    pickle.dump(encoders, f)

print("Saved: heart_model.pkl, heart_scaler.pkl, heart_encoders.pkl")

---
## 3️⃣ Parkinson's Disease Prediction

**Dataset:** Oxford Parkinson's Disease Detection Dataset (195 voice recordings, 22 biomedical voice features)
**Source URL:** `https://archive.ics.uci.edu/ml/machine-learning-databases/parkinsons/parkinsons.data`
**Target:** `status` → 0 = Healthy, 1 = Parkinson's


In [ ]:
# ----- 3.1 Load Data -----
parkinsons_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/parkinsons/parkinsons.data"
parkinsons_df = pd.read_csv(parkinsons_url)
print("Shape:", parkinsons_df.shape)
parkinsons_df.head()

In [ ]:
# ----- 3.2 Exploratory Data Analysis -----
parkinsons_df.info()
print("\nMissing values:\n", parkinsons_df.isnull().sum().sum(), "total missing values")
print("\nClass balance:\n", parkinsons_df['status'].value_counts())

# Drop the 'name' column (voice-recording identifier, not a predictive feature)
parkinsons_clean = parkinsons_df.drop(columns=['name'])

In [ ]:
# ----- 3.3 Visualizations -----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(x='status', data=parkinsons_clean, ax=axes[0])
axes[0].set_title("Parkinson's Outcome Distribution")

sns.heatmap(parkinsons_clean.corr(), cmap="coolwarm", ax=axes[1])
axes[1].set_title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
# ----- 3.4 Train/Test Split & Scaling -----
X_park = parkinsons_clean.drop('status', axis=1)
y_park = parkinsons_clean['status']

X_park_train, X_park_test, y_park_train, y_park_test = train_test_split(
    X_park, y_park, test_size=0.2, random_state=RANDOM_STATE, stratify=y_park)

scaler_park = StandardScaler()
X_park_train_s = scaler_park.fit_transform(X_park_train)
X_park_test_s = scaler_park.transform(X_park_test)

print("Train size:", X_park_train.shape, "| Test size:", X_park_test.shape)

In [ ]:
# ----- 3.5 Train & Compare Multiple Models -----
park_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE),
    "SVM": SVC(probability=True, random_state=RANDOM_STATE),
    "KNN": KNeighborsClassifier(),
}

park_results = {}
for name, model in park_models.items():
    model.fit(X_park_train_s, y_park_train)
    preds = model.predict(X_park_test_s)
    acc = accuracy_score(y_park_test, preds)
    park_results[name] = acc
    print(f"{name:20s} -> Accuracy: {acc:.4f}")

best_park_name = max(park_results, key=park_results.get)
best_park_model = park_models[best_park_name]
print(f"\n🏆 Best Model: {best_park_name} ({park_results[best_park_name]:.4f})")

In [ ]:
# ----- 3.6 Evaluate Best Model -----
park_preds = best_park_model.predict(X_park_test_s)
print(classification_report(y_park_test, park_preds, target_names=["Healthy", "Parkinson's"]))

cm = confusion_matrix(y_park_test, park_preds)
ConfusionMatrixDisplay(cm, display_labels=["Healthy", "Parkinson's"]).plot(cmap="Purples")
plt.title(f"Confusion Matrix - {best_park_name} (Parkinson's)")
plt.show()

In [ ]:
# ----- 3.7 Save Model -----
with open("parkinsons_model.pkl", "wb") as f:
    pickle.dump(best_park_model, f)
with open("parkinsons_scaler.pkl", "wb") as f:
    pickle.dump(scaler_park, f)

print("Saved: parkinsons_model.pkl, parkinsons_scaler.pkl")

---
## 4️⃣ Summary & Unified Prediction Function

A quick comparison of best accuracy per disease, plus simple helper functions you can call directly with new patient data (values must be in the same order as the original feature columns).


In [ ]:
# ----- 4.1 Results Summary -----
summary = pd.DataFrame({
    "Disease": ["Diabetes", "Heart Disease", "Parkinson's"],
    "Best Model": [best_dia_name, best_heart_name, best_park_name],
    "Test Accuracy": [dia_results[best_dia_name], heart_results[best_heart_name], park_results[best_park_name]]
})
summary

In [ ]:
# ----- 4.2 Unified Prediction Helpers -----
def predict_diabetes(input_values):
    """input_values: list in order = [Pregnancies, Glucose, BloodPressure, SkinThickness,
                                        Insulin, BMI, DiabetesPedigreeFunction, Age]"""
    arr = np.array(input_values).reshape(1, -1)
    arr_scaled = scaler_dia.transform(arr)
    pred = best_dia_model.predict(arr_scaled)[0]
    return "Diabetic" if pred == 1 else "Not Diabetic"


def predict_heart_disease(input_values):
    """input_values: list in order matching X_heart.columns (categorical fields must
    already be label-encoded using heart_encoders.pkl before calling this function)"""
    arr = np.array(input_values).reshape(1, -1)
    arr_scaled = scaler_heart.transform(arr)
    pred = best_heart_model.predict(arr_scaled)[0]
    return "Heart Disease Detected" if pred == 1 else "No Heart Disease"


def predict_parkinsons(input_values):
    """input_values: list in order matching X_park.columns (22 voice-measurement features)"""
    arr = np.array(input_values).reshape(1, -1)
    arr_scaled = scaler_park.transform(arr)
    pred = best_park_model.predict(arr_scaled)[0]
    return "Parkinson's Detected" if pred == 1 else "Healthy"


# Example usage (replace with real patient data):
sample_diabetes_input = X_dia_test.iloc[0].values
print("Diabetes example prediction:", predict_diabetes(sample_diabetes_input))

sample_heart_input = X_heart_test.iloc[0].values
print("Heart example prediction:", predict_heart_disease(sample_heart_input))

sample_park_input = X_park_test.iloc[0].values
print("Parkinson's example prediction:", predict_parkinsons(sample_park_input))

---
## ✅ Next Steps

- **Deploy as a web app:** wrap the three `predict_*` functions in a Flask or Streamlit app with input forms for each disease.
- **Improve accuracy:** try hyperparameter tuning (`GridSearchCV`), feature selection, or ensembling (e.g. XGBoost, stacking).
- **Handle class imbalance:** if a dataset is imbalanced, try `SMOTE` (from `imblearn`) or class-weighted models.
- **Explainability:** add SHAP or feature-importance plots so predictions are interpretable for a medical audience.
- **Packaging for FYP/portfolio:** this notebook, together with the six `.pkl` files it produces, is enough to build a full end-to-end multi-disease prediction system (matches the popular "Multiple Disease Prediction System" project pattern used in many FYPs).
